In [1]:
import http.client
import json

insights = ["tariffs", "deepseek", "luka doncic", "stock market", "apple"]
# insights = ["bitcoin", ]

conn = http.client.HTTPSConnection("google.serper.dev")
payload = json.dumps([
    {
        "q": insight,
        "num": 10,
        "tbs": "qdr:d"
    } for insight in insights
])
headers = {
  'X-API-KEY': '39ef0015c9282897135dcf73ee553d994cfb895d',
  'Content-Type': 'application/json'
}

In [2]:
print(payload)

[{"q": "tariffs", "num": 10, "tbs": "qdr:d"}, {"q": "deepseek", "num": 10, "tbs": "qdr:d"}, {"q": "luka doncic", "num": 10, "tbs": "qdr:d"}, {"q": "stock market", "num": 10, "tbs": "qdr:d"}, {"q": "apple", "num": 10, "tbs": "qdr:d"}]


In [3]:
conn.request("POST", "/news", payload, headers)
res = conn.getresponse()
data = res.read()
print(data.decode("utf-8"))

[{"searchParameters":{"q":"tariffs","type":"news","num":10,"tbs":"qdr:d","engine":"google"},"news":[{"title":"Trump Tariffs Live Updates: U.S. and Mexico Reach Deal to Delay Tariffs","link":"https://www.nytimes.com/live/2025/02/03/us/trump-tariffs","snippet":"President Trump said he would pause tariffs on Mexico for a month, but levies on Canada and China were still set to take effect on Tuesday.","date":"33 minutes ago","source":"The New York Times","imageUrl":"https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR4rSr2h8D8nlad3FnV0ZF9QzRQrHyOQaacfNdjVDEXQnu8_FlOMyCTJF4&usqp=CAI&s","position":1},{"title":"Canada’s most populous province to pause retaliatory measures as US puts tariffs on hold","link":"https://apnews.com/article/canada-trump-tariffs-ontario-musk-2bc1b52b0390aee9686b270606c51573","snippet":"TORONTO (AP) — The leader of Ontario, Canada's most populous province, said Monday he will pause all retaliatory measures against the United States after...","date":"7 hours ago","s

In [4]:
# Parse the JSON data
parsed_data = json.loads(data.decode("utf-8"))

# Pretty print the data with indentation
print(json.dumps(parsed_data, indent=2))


[
  {
    "searchParameters": {
      "q": "tariffs",
      "type": "news",
      "num": 10,
      "tbs": "qdr:d",
      "engine": "google"
    },
    "news": [
      {
        "title": "Trump Tariffs Live Updates: U.S. and Mexico Reach Deal to Delay Tariffs",
        "link": "https://www.nytimes.com/live/2025/02/03/us/trump-tariffs",
        "snippet": "President Trump said he would pause tariffs on Mexico for a month, but levies on Canada and China were still set to take effect on Tuesday.",
        "date": "33 minutes ago",
        "source": "The New York Times",
        "imageUrl": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR4rSr2h8D8nlad3FnV0ZF9QzRQrHyOQaacfNdjVDEXQnu8_FlOMyCTJF4&usqp=CAI&s",
        "position": 1
      },
      {
        "title": "Canada\u2019s most populous province to pause retaliatory measures as US puts tariffs on hold",
        "link": "https://apnews.com/article/canada-trump-tariffs-ontario-musk-2bc1b52b0390aee9686b270606c51573",
        "snippe

In [5]:
import requests
from bs4 import BeautifulSoup

def get_high_res_image(page_url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0",
            # Only request the HTML, not images/css/js
            "Accept": "text/html",
            # Allow compressed responses
            "Accept-Encoding": "gzip, deflate"
        }
        # Add timeout to avoid hanging on slow responses
        response = requests.get(page_url, headers=headers)
        
        # Parse only the head section where meta tags usually are
        head_content = response.text.split('</head>')[0] + '</head>'
        soup = BeautifulSoup(head_content, 'html.parser')

        # Try to find OpenGraph image first (most common and fastest check)
        og_image = soup.find("meta", property="og:image")
        if og_image and og_image.get("content"):
            return og_image["content"]

        # If no OG image, only then parse the full body
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Look for first image with srcset
        img_with_srcset = soup.find("img", srcset=True)
        if img_with_srcset:
            return img_with_srcset["srcset"].split(",")[-1].split(" ")[0]

        # Last resort: first image
        first_img = soup.find("img", src=True)
        return first_img["src"] if first_img else None

    except Exception:
        return None

In [6]:
import pandas as pd

# Initialize lists to store the data
titles = []
links = []
snippets = []
dates = []
sources = []
query_terms = []
image_urls = []

# Iterate through each query result
for i, query_result in enumerate(parsed_data):
    for article in query_result.get('news', []):
        titles.append(article.get('title', ''))
        links.append(article.get('link', ''))
        snippets.append(article.get('snippet', ''))
        dates.append(article.get('date', ''))
        sources.append(article.get('source', ''))
        query_terms.append(insights[i])
        image_urls.append(get_high_res_image(article.get('link', '')))

# Create the DataFrame
df = pd.DataFrame({
    # 'query_term': query_terms,
    'title': titles,
    'link': links,
    'snippet': snippets,
    'time_scraped': dates,
    'query_term': query_terms,
    'image_url': image_urls,
    'source': sources
})

# Display the first few rows
display(df)


,title,link,snippet,time_scraped,query_term,image_url,source
0,Trump Tariffs Live Updates: U.S. and Mexico Re...,https://www.nytimes.com/live/2025/02/03/us/tru...,President Trump said he would pause tariffs on...,33 minutes ago,tariffs,None,The New York Times
1,Canada’s most populous province to pause retal...,https://apnews.com/article/canada-trump-tariff...,"TORONTO (AP) — The leader of Ontario, Canada's...",7 hours ago,tariffs,https://dims.apnews.com/dims4/default/e20870e/...,AP News
2,"Mexico, Canada avert Trump tariffs; Musk’s rol...",https://www.aljazeera.com/news/liveblog/2025/2...,Pause on tariffs offers brief reprieve as Demo...,LIVE47 minutes ago,tariffs,https://www.aljazeera.com/wp-content/uploads/2...,Al Jazeera
3,Tariffs latest: Trump says planned US sovereig...,https://www.ft.com/content/628f137c-f950-4731-...,The EU is “prepared” to respond if US Presiden...,LIVE43 minutes ago,tariffs,https://www.ft.com/__origami/service/image/v2/...,Financial Times
4,Live updates: Trump pauses tariffs on Mexico a...,https://www.nbcnews.com/politics/politics-news...,Follow the latest news on President Donald Tru...,LIVE14 minutes ago,tariffs,https://media-cldnry.s-nbcnews.com/image/uploa...,NBC News
5,Trump agrees to pause tariffs on Canada and Me...,https://www.bbc.com/news/articles/c87d5rlee52o,President Donald Trump has agreed to hold off ...,2 hours ago,tariffs,https://ichef.bbci.co.uk/news/1024/branded_new...,BBC
6,Trump’s 25% tariffs on Canada and Mexico will ...,https://www.brookings.edu/articles/trumps-25-t...,Joshua P. Meltzer analyzes the effects of tari...,5 hours ago,tariffs,https://www.brookings.edu/wp-content/uploads/2...,Brookings
7,Trump to pause Canada tariffs for at least 30 ...,https://www.theguardian.com/us-news/live/2025/...,Canadian PM Justin Trudeau says he had 'good c...,LIVE1 hour ago,tariffs,https://i.guim.co.uk/img/media/6c6c32f7de6d466...,The Guardian
8,"Trump pauses tariffs on Mexico and Canada, but...",https://www.reuters.com/world/us/trump-says-am...,Both Canadian Prime Minister Trudeau and Mexic...,4 hours ago,tariffs,None,Reuters
9,"February 3, 2025: Donald Trump presidency news",https://www.cnn.com/politics/live-news/trump-t...,President Donald Trump said he agreed to “imme...,LIVE2 hours ago,tariffs,https://media.cnn.com/api/v1/images/stellar/pr...,CNN


In [7]:
from dotenv import load_dotenv
import os
load_dotenv()

from supabase import create_client, Client

supabase: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))


In [8]:
# Convert DataFrame to records and upsert based on title and source as unique identifiers
supabase.table("Articles").insert(
    df.to_dict(orient="records")
).execute()

APIResponse[~_ReturnT](data=[{'id': 151, 'title': 'Trump Tariffs Live Updates: U.S. and Mexico Reach Deal to Delay Tariffs', 'link': 'https://www.nytimes.com/live/2025/02/03/us/trump-tariffs', 'snippet': 'President Trump said he would pause tariffs on Mexico for a month, but levies on Canada and China were still set to take effect on Tuesday.', 'time_scraped': '33 minutes ago', 'source': 'The New York Times', 'created_at': '2025-02-04T05:18:44.714239+00:00', 'image_url': None, 'query_term': 'tariffs'}, {'id': 152, 'title': 'Canada’s most populous province to pause retaliatory measures as US puts tariffs on hold', 'link': 'https://apnews.com/article/canada-trump-tariffs-ontario-musk-2bc1b52b0390aee9686b270606c51573', 'snippet': "TORONTO (AP) — The leader of Ontario, Canada's most populous province, said Monday he will pause all retaliatory measures against the United States after...", 'time_scraped': '7 hours ago', 'source': 'AP News', 'created_at': '2025-02-04T05:18:44.714239+00:00', '